# 07 · Apache Spark in a notebook
Needs Java 17+ and `pip install pyspark`. Same questions as SQL Lab, on a distributed engine (running locally here).

In [ ]:
import sys
sys.path.insert(0, "..")
from common import *
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder.appName("bi-notebook").master("local[*]")
         .config("spark.ui.showConsoleProgress", "false").config("spark.sql.shuffle.partitions", "4").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
sdf = spark.read.csv(str(ensure_sample()), header=True, inferSchema=True).withColumn("profit", F.col("revenue") - F.col("operational_cost"))
sdf.printSchema()

## DataFrame API and Spark SQL are two spellings of one plan

In [ ]:
(sdf.groupBy("category").agg(F.round(F.sum("revenue"), 2).alias("revenue"),
                              F.round(F.sum("profit") / F.sum("revenue") * 100, 2).alias("margin_pct"))
     .orderBy(F.desc("revenue")).show())

In [ ]:
sdf.createOrReplaceTempView("ops")
spark.sql("SELECT category, ROUND(SUM(revenue), 2) AS revenue FROM ops GROUP BY category ORDER BY revenue DESC").show()

Spark is **lazy**: transformations only build a plan; `.show()`, `.count()` and `.write` run it. `.explain()` prints the plan.

In [ ]:
sdf.groupBy("category").agg(F.sum("revenue")).explain()

## Your turn
Which **entity_name** has the highest total profit? Store its name (a string) in `top_entity`, using Spark (not pandas).

In [ ]:
top_entity = None

In [ ]:
import pandas as pd
p = pd.read_csv(ensure_sample()); p["profit"] = p["revenue"] - p["operational_cost"]
check("top entity by profit", top_entity == p.groupby("entity_name")["profit"].sum().idxmax(), "groupBy('entity_name').agg(F.sum('profit')).orderBy(F.desc(...)).first()")

In [ ]:
spark.stop()